# Cluster Control Panel

Пульт управления кластером федеративного обучения.
Запускать из корня проекта (`ddl/`).

| # | Модуль | Описание |
|---|--------|----------|
| 1 | **Проверка узлов** | SSH-доступность, окружение, датасеты |
| 2 | **Датасет** | Скачивание и просмотр структуры |
| 3 | **Распределение данных** | Партиционирование и rsync на узлы |
| 4 | Запуск кластера | SuperLink + SuperNodes |
| 5 | Мониторинг | Метрики обучения в реальном времени |

## Setup
Загрузка конфигурации узлов из `deploy/nodes.conf`.

In [ ]:
import subprocess
import concurrent.futures
import sys

import pandas as pd
from IPython.display import display

sys.path.insert(0, ".")
from scripts.cluster_utils import load_nodes, setup_ssh_key, ssh_node

cfg = load_nodes()
setup_ssh_key(cfg)  # копируем ключ из Windows → WSL один раз явно

print(f"Сервер : {cfg['server']['name']} ({cfg['server']['ip']})")
print(f"Клиенты: {len(cfg['clients'])}")
for c in cfg["clients"]:
    print(f"  partition-id={c['partition_id']}: {c['name']}  внешний={c['ext_ip']}  внутренний={c['ip']}")


---
## 1. Проверка узлов

Для каждого узла проверяем:
- **SSH** — доступность по ключу
- **venv** — наличие виртуального окружения
- **flwr** — версия Flower
- **dataset** — наличие папки с данными (только клиенты)

In [ ]:
def check_node(node: dict) -> dict:
    """Проверяет готовность одного узла. Возвращает dict со статусами."""
    remote_dir = cfg["remote_dir"]

    def ssh(cmd):
        r = ssh_node(cfg, node, cmd, timeout=20)
        return r.returncode == 0, r.stdout.strip()

    row = {
        "Узел":         node["name"],
        "Внешний IP":   node["ext_ip"],
        "partition-id": node["partition_id"] if node["partition_id"] is not None else "—",
        "SSH":     "✗",
        "venv":    "—",
        "flwr":    "—",
        "dataset": "—",
    }

    ok, _ = ssh("echo pong")
    if not ok:
        return row
    row["SSH"] = "✓"

    ok, _ = ssh(f"test -f {remote_dir}/.venv/bin/activate && echo yes")
    row["venv"] = "✓" if ok else "✗"

    ok, out = ssh(f"source {remote_dir}/.venv/bin/activate 2>/dev/null && "
                  f"python -c 'import flwr; print(flwr.__version__)' 2>/dev/null")
    row["flwr"] = out if ok and out else "✗"

    if node["role"] == "client":
        ok, _ = ssh(f"test -d {remote_dir}/data && echo yes")
        row["dataset"] = "✓" if ok else "✗"

    return row


print("Проверяем узлы (параллельно)...")
with concurrent.futures.ThreadPoolExecutor() as ex:
    rows = list(ex.map(check_node, cfg["all_nodes"]))

rows.sort(key=lambda r: (r["Узел"] != "fl-server", r["Узел"]))
df = pd.DataFrame(rows).set_index("Узел")
display(df)

total = len(rows)
ready = sum(1 for r in rows if r["SSH"] == "✓" and r["venv"] == "✓" and r["flwr"] != "✗")
print(f"\nГотово узлов: {ready}/{total}")
if ready < total:
    print("Запусти deploy/deploy_all.sh чтобы настроить недостающие узлы.")


### 1.1 Деплой на все узлы

Rsync кода + установка окружения (`python3.12`, `venv`, `flwr`, датасеты) на все VM.  
Запускай если новые узлы не имеют `venv` или `flwr` по результатам проверки выше.

In [ ]:
# 1.1 — Деплой кода и установка окружения на все VM
import subprocess, sys

proc = subprocess.Popen(
    ["bash", "deploy/deploy_all.sh"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)
for line in proc.stdout:
    print(line, end="", flush=True)
proc.wait()
print("Exit code:", proc.returncode)

---
## 2. Датасет

Скачивает датасет (если ещё нет локально) и показывает его структуру:
сплиты, количество примеров, названия классов и их распределение.

In [ ]:
import collections
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np
from pathlib import Path
from datasets import load_from_disk

from scripts.download_datasets import download_and_save

# ==============================================================================
# Укажи датасет для скачивания.
#
# Стандартные (HuggingFace Hub):
#   "cifar10"          — CIFAR-10  (32x32, 10 классов,  60k изображений)
#   "cifar100"         — CIFAR-100 (32x32, 100 классов, 60k изображений)
#   "mnist"            — MNIST (28x28, цифры 0-9, 70k изображений)
#   "fashion_mnist"    — Fashion-MNIST (28x28, одежда, 70k изображений)
#   "svhn"             — Street View House Numbers
#   "food101"          — Food-101 (101 класс еды)
#   "beans"            — болезни растений (3 класса)
#   "author/dataset"   — любой публичный датасет с HF Hub
#
# Kaggle (скачиваются отдельным скриптом):
#   python -m scripts.download_datasets --only plantvillage
#   → сохраняется в data/plantvillage/  — указывай DATASET = "plantvillage" и LOCAL_PATH ниже
#
# Найти датасеты: https://huggingface.co/datasets?task_categories=image-classification
# ==============================================================================
DATASET    = "cifar100"
DATA_DIR   = Path("data/")
LOCAL_PATH = DATA_DIR / DATASET   # для kaggle-датасетов можно задать вручную, например DATA_DIR / "plantvillage"
# ==============================================================================

print(f"Датасет : {DATASET}")
print(f"Путь    : {LOCAL_PATH.resolve()}")

download_and_save(DATASET, LOCAL_PATH)

ds = load_from_disk(str(LOCAL_PATH))
train = ds["train"]

# --- Сплиты и размеры ---
print(f"\n{'Сплит':<10} {'Примеров':>10}")
print("-" * 22)
for split, data in ds.items():
    print(f"{split:<10} {len(data):>10,}")

# --- Поля ---
print(f"\nПоля: {list(train.features.keys())}")

# --- Классы ---
label_feat = train.features.get("label") or train.features.get("labels") or train.features.get("fine_label") or train.features.get("coarse_label")
if label_feat and hasattr(label_feat, "names"):
    class_names = label_feat.names
    print(f"\nКлассов: {len(class_names)}")
    print(f"\n{'idx':>4}  Название")
    print("-" * 20)
    for idx, name in enumerate(class_names):
        print(f"  {idx:>3}  {name}")
else:
    class_names = [str(i) for i in sorted(set(train["label"]))]
    print(f"\nНазвания классов не найдены в метаданных, используем индексы: {class_names}")


---
## 3. Распределение данных

Разбивает тренировочный датасет на партиции и рассылает их по клиентам.

**Алгоритм** `floor + scheme`:
1. Из каждого класса берём `MIN_PER_CLASS` образцов на клиента — гарантированный минимум.
2. Оставшиеся образцы распределяем по схеме: `iid` (поровну) или `dirichlet` (пропорционально Dir(α)).

**Ячейка 3.1** — конфиг + партиционирование + визуализация  
**Ячейка 3.2** — rsync партиций на клиентов и тест-сплита на сервер

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datasets import load_from_disk
from IPython.display import display

from scripts.partition_utils import (
    extract_server_dataset, partition_dataset, save_partitions,
    partition_dir_name, load_manifest,
)
from scripts.download_datasets import download_and_save

# ==============================================================================
# Параметры разбивки — меняй здесь
# ==============================================================================
DATASET       = "cifar10"    # датасет (должен совпадать с тем, что скачан в ячейке 2)
NUM_CLIENTS   = 6            # число клиентов (должно совпадать с кластером)
SCHEME        = "dirichlet"  # "iid" или "dirichlet"
ALPHA         = 0.7          # параметр Dirichlet (меньше → сильнее non-IID; игнорируется при iid)
MIN_PER_CLASS = 50           # гарантированный минимум образцов каждого класса у каждого клиента
SERVER_SIZE   = 0            # образцов в серверном датасете (0 — отключено); делится поровну по классам
SEED          = 42
FORCE         = False        # True — пересчитать партиции даже если уже существуют
# ==============================================================================

DATA_DIR   = Path("data/")
local_path = DATA_DIR / DATASET
part_name  = partition_dir_name(DATASET, SCHEME, NUM_CLIENTS, SEED,
                                alpha=ALPHA, min_per_class=MIN_PER_CLASS,
                                server_size=SERVER_SIZE)
out_dir    = DATA_DIR / "partitions" / part_name

print(f"Датасет      : {DATASET}")
print(f"Схема        : {SCHEME}" + (f"  (alpha={ALPHA})" if SCHEME == "dirichlet" else ""))
print(f"Клиентов     : {NUM_CLIENTS}")
print(f"Min/class    : {MIN_PER_CLASS}")
if SERVER_SIZE > 0:
    print(f"Серверный    : {SERVER_SIZE} образцов (сбалансированный)")
print(f"Seed         : {SEED}")
print(f"Директория   : {out_dir}")
print()

# Скачиваем датасет, если ещё нет
download_and_save(DATASET, local_path)
ds = load_from_disk(str(local_path))

# Партиционирование
if out_dir.exists() and not FORCE:
    print(f"Партиции уже существуют: {out_dir}")
    print("Загружаем manifest.json...")
    manifest = load_manifest(out_dir)
else:
    print("Разбиваем датасет...")
    train_ds = ds["train"]

    # Сначала выделяем серверный датасет (если задан)
    server_ds = None
    if SERVER_SIZE > 0:
        server_ds, train_ds = extract_server_dataset(train_ds, SERVER_SIZE, seed=SEED)
        print(f"  Серверный датасет: {len(server_ds):,} образцов")
        print(f"  Осталось для клиентов: {len(train_ds):,} образцов")

    partitions = partition_dataset(
        train_ds,
        num_clients=NUM_CLIENTS,
        scheme=SCHEME,
        alpha=ALPHA,
        min_per_class=MIN_PER_CLASS,
        seed=SEED,
    )
    out_dir = save_partitions(
        ds, partitions, out_dir,
        dataset=DATASET, scheme=SCHEME, alpha=ALPHA,
        min_per_class=MIN_PER_CLASS, seed=SEED,
        server_dataset=server_ds, force=FORCE,
    )
    manifest = load_manifest(out_dir)

# ── Таблица: классы × клиенты ─────────────────────────────────────────────
class_names  = manifest["class_names"]
num_classes  = manifest["num_classes"]
num_clients  = manifest["num_clients"]

counts_matrix = np.array([
    [manifest["clients"][i]["classes"].get(str(c), 0) for i in range(num_clients)]
    for c in range(num_classes)
])

table_data = {f"client_{i}": counts_matrix[:, i] for i in range(num_clients)}
df = pd.DataFrame(table_data, index=class_names)
display(df)

totals = [manifest["clients"][i]["total"] for i in range(num_clients)]
print(f"\nОбразцов на клиента: min={min(totals):,}  max={max(totals):,}  avg={sum(totals)//len(totals):,}")
srv_size = manifest.get("server_size")
srv_info = f"  |  server: {srv_size:,}" if srv_size else ""
print(f"Всего train: {sum(totals):,}{srv_info}  |  test: {manifest.get('test_size') or '—'}")

if SCHEME == "dirichlet":
    # Показываем фактический non-IID "перекос" по классам:
    # для каждого класса — доля образцов у самого богатого клиента
    max_shares = counts_matrix.max(axis=1) / counts_matrix.sum(axis=1)
    print(f"\nNon-IID: доля самого богатого клиента по каждому классу (идеал IID = {1/num_clients:.2f}):")
    for name, share in zip(class_names, max_shares):
        bar = "█" * int(share * 30)
        print(f"  {name:>12}: {share:.2f}  {bar}")

# ── Heatmap ───────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(max(6, num_clients * 1.4), max(4, num_classes * 0.55)))
im = ax.imshow(counts_matrix, aspect="auto", cmap="YlOrRd")
plt.colorbar(im, ax=ax, label="Образцов")
ax.set_xticks(range(num_clients))
ax.set_xticklabels([f"client_{i}" for i in range(num_clients)], rotation=45, ha="right")
ax.set_yticks(range(num_classes))
ax.set_yticklabels(class_names, fontsize=8)
ax.set_xlabel("Клиент")
ax.set_ylabel("Класс")
title = f"{DATASET} / {SCHEME}"
if SCHEME == "dirichlet":
    title += f"  α={ALPHA}  min/class={MIN_PER_CLASS}"
if SERVER_SIZE > 0:
    title += f"  server={SERVER_SIZE}"
ax.set_title(title)

for c in range(num_classes):
    for i in range(num_clients):
        ax.text(i, c, str(counts_matrix[c, i]), ha="center", va="center", fontsize=7)

plt.tight_layout()
plt.show()

In [ ]:
# 3.2 — Удаление полного датасета с клиентов
# Удаляем {REMOTE_DIR}/data/{DATASET}/ на каждом клиенте.
# Партиции (data/partitions/) не трогаем.
import concurrent.futures
from scripts.cluster_utils import ssh_client
# Задайте явное имя датасета
DATASET = 'mnist'

def delete_dataset_on_client(node):
    path = f"{cfg['remote_dir']}/data/{DATASET}"
    r = ssh_client(cfg, node["ip"],
                   f"rm -rf {path} && echo 'deleted' || echo 'not found'",
                   timeout=30)
    status = r.stdout.strip() if r.returncode == 0 else f"FAILED: {r.stderr.strip()}"
    return node["name"], status

print(f"Удаляем {DATASET}/ на всех клиентах (параллельно)...")
with concurrent.futures.ThreadPoolExecutor() as ex:
    results = list(ex.map(delete_dataset_on_client, cfg["clients"]))

for name, status in results:
    icon = "✓" if "FAILED" not in status else "✗"
    print(f"  {icon} {name}: {status}")

In [ ]:
# 3.3 — Rsync партиций на клиентов и тест-сплита на сервер
import json
from pathlib import Path
from scripts.cluster_utils import ssh_client, rsync_to_client, ssh_server, rsync_to_server
from scripts.partition_utils import load_manifest

# ── ЯВНОЕ имя партиции ────────────────────────────────────────
PART_NAME = "cifar10__dirichlet__a0.8__m50__n6__s42"   # ← оставь пустым чтобы взять из ячейки 3.1, или задай явно

if PART_NAME:
    part_name = PART_NAME

REMOTE_PART_DIR = f"{cfg['remote_dir']}/data/partitions/{part_name}"
out_dir = Path("data/partitions") / part_name

# Загружаем manifest (из памяти если есть, иначе с диска)
try:
    manifest
except NameError:
    manifest = load_manifest(out_dir)
    print(f"manifest загружен с диска: {out_dir}/manifest.json")

print(f"Партиция : {part_name}")
print(f"Удалённая директория: {REMOTE_PART_DIR}\n")

# ── Умное назначение: большие партиции → быстрые узлы ────────────────────
part_sizes = [(i, manifest["clients"][i]["total"]) for i in range(manifest["num_clients"])]
part_sizes.sort(key=lambda x: x[1], reverse=True)

nodes_sorted = sorted(cfg["clients"], key=lambda n: (-n["cores"], n["partition_id"]))

assignment = []  # (node, src_part_id, dst_part_id, size)
for (src_id, size), node in zip(part_sizes, nodes_sorted):
    dst_id = node["partition_id"]
    assignment.append((node, src_id, dst_id, size))

print(f"{'Узел':<14} {'Ядер':>5}  {'src':>8}  {'dst':>8}  {'Примеров':>10}")
print("─" * 56)
for node, src_id, dst_id, size in assignment:
    marker = "  ←" if src_id != dst_id else ""
    print(f"  {node['name']:<12} {node['cores']:>5}  client_{src_id}  →  client_{dst_id}  {size:>10,}{marker}")
print()

errors = []

# ── Клиенты: rsync src → remote как dst ──────────────────────────────────
for node, src_id, dst_id, size in assignment:
    src = str(out_dir / f"client_{src_id}") + "/"
    dst = f"{REMOTE_PART_DIR}/client_{dst_id}/"

    r = ssh_client(cfg, node["ip"], f"mkdir -p {REMOTE_PART_DIR}/", timeout=15)
    if r.returncode != 0:
        print(f"  ✗ {node['name']} mkdir FAILED: {(r.stderr or '').strip()}")
        errors.append(node["name"])
        continue

    r = rsync_to_client(cfg, node["ip"], src, dst, timeout=180)
    if r.returncode == 0:
        print(f"  ✓ {node['name']}  client_{src_id} → client_{dst_id}  ({size:,} samples)")
    else:
        print(f"  ✗ {node['name']} rsync FAILED: {(r.stderr or '').strip()}")
        errors.append(node["name"])

# ── Серверный датасет → все клиенты + сервер (если существует) ───────────
server_dir = out_dir / "server"
if server_dir.exists():
    srv_size = manifest.get("server_size", "?")
    print(f"\nСерверный датасет ({srv_size} образцов) → все клиенты + сервер...")
    for node in cfg["clients"]:
        src = str(server_dir) + "/"
        dst = f"{REMOTE_PART_DIR}/server/"
        r = rsync_to_client(cfg, node["ip"], src, dst, timeout=120)
        if r.returncode == 0:
            print(f"  ✓ {node['name']}  server/ ({srv_size} samples)")
        else:
            print(f"  ✗ {node['name']} server/ rsync FAILED: {(r.stderr or '').strip()}")
            errors.append(f"{node['name']}/server")

    # Сервер тоже нужен — server_app.py читает server/ для построения расписания
    ssh_server(cfg, f"mkdir -p {REMOTE_PART_DIR}/server/", timeout=15)
    r = rsync_to_server(cfg, str(server_dir) + "/", f"{REMOTE_PART_DIR}/server/", timeout=120)
    if r.returncode == 0:
        print(f"  ✓ fl-server  server/ ({srv_size} samples)")
    else:
        print(f"  ✗ fl-server server/ rsync FAILED: {(r.stderr or '').strip()}")
        errors.append("fl-server/server")
else:
    print("\n  (server/ не найден — пропускаем)")

# ── Сервер: тест-сплит + manifest.json ───────────────────────────────────
r = ssh_server(cfg, f"mkdir -p {REMOTE_PART_DIR}/", timeout=15)
if r.returncode != 0:
    print(f"  ✗ fl-server mkdir FAILED: {(r.stderr or '').strip()}")
    errors.append("fl-server")
else:
    test_src = str(out_dir / "test") + "/"
    r = rsync_to_server(cfg, test_src, f"{REMOTE_PART_DIR}/test/", timeout=180)
    if r.returncode == 0:
        print(f"  ✓ fl-server  test  ({manifest.get('test_size', '?')} samples)")
    else:
        print(f"  ✗ fl-server test rsync FAILED: {(r.stderr or '').strip()}")
        errors.append("fl-server/test")

    manifest_src = str(out_dir / "manifest.json")
    r = rsync_to_server(cfg, manifest_src, f"{REMOTE_PART_DIR}/manifest.json", timeout=30)
    if r.returncode == 0:
        print(f"  ✓ fl-server  manifest.json")
    else:
        print(f"  ✗ fl-server manifest rsync FAILED: {(r.stderr or '').strip()}")
        errors.append("fl-server/manifest")

print()
if not errors:
    print("✓ Все партиции разосланы. Назначение оптимизировано по ядрам CPU.")
else:
    print(f"✗ Ошибки на: {', '.join(errors)}")

In [ ]:
# 3.4 — Инвентаризация партиций на клиентах
import concurrent.futures
import json
import pandas as pd
from IPython.display import display
from scripts.cluster_utils import ssh_client

# load_from_disk через venv — Arrow memory-mapped, len() это O(1)
CHECK_CMD = r"""bash -c 'source ~/ddl/.venv/bin/activate && python3 << PYEOF
import json, os
from datasets import load_from_disk

parts_dir = os.path.expanduser("~/ddl/data/partitions")
result = {}
if os.path.isdir(parts_dir):
    for part_name in sorted(os.listdir(parts_dir)):
        part_path = os.path.join(parts_dir, part_name)
        if not os.path.isdir(part_path):
            continue
        for entry in sorted(os.listdir(part_path)):
            if not entry.startswith("client_"):
                continue
            try:
                d = load_from_disk(os.path.join(part_path, entry))
                result[part_name + "/" + entry] = len(d)
            except Exception as e:
                result[part_name + "/" + entry] = -1
print(json.dumps(result))
PYEOF'"""


def scan_client(node):
    r = ssh_client(cfg, node["ip"], CHECK_CMD, timeout=60)
    if r.returncode != 0:
        return node["name"], None
    try:
        return node["name"], json.loads(r.stdout.strip())
    except (json.JSONDecodeError, ValueError):
        return node["name"], {}


print("Сканируем клиентов (параллельно)...")
with concurrent.futures.ThreadPoolExecutor() as ex:
    scan_results = list(ex.map(scan_client, cfg["clients"]))

client_names = [n["name"] for n in cfg["clients"]]

# Собираем все уникальные имена партиций
all_partitions: set[str] = set()
for _, data in scan_results:
    if data:
        for key in data:
            all_partitions.add(key.rsplit("/", 1)[0])

if not all_partitions:
    print("Партиции не найдены ни на одном клиенте.")
else:
    table: dict[str, dict[str, str]] = {p: {} for p in sorted(all_partitions)}

    for client_name, data in scan_results:
        if data is None:
            for p in table:
                table[p][client_name] = "SSH ERR"
            continue
        for key, count in data.items():
            part_name = key.rsplit("/", 1)[0]
            table[part_name][client_name] = f"{count:,}" if count >= 0 else "load err"

    for p in table:
        for cn in client_names:
            table[p].setdefault(cn, "—")

    df = pd.DataFrame(table, index=client_names).T
    df.index.name = "Партиция"
    display(df)

    for p, row in table.items():
        nums = [int(v.replace(",", "")) for v in row.values() if v.replace(",", "").isdigit()]
        print(f"  {p}: {sum(nums):,} образцов суммарно")

In [ ]:
# 3.5 — Удаление партиции с клиентов (и сервера)
import concurrent.futures
from scripts.cluster_utils import ssh_client, ssh_server

# Укажи имя партиции для удаления (можно скопировать из таблицы выше)
DELETE_PARTITION = "cifar10__iid__n5__s42"

remote_path = f"{cfg['remote_dir']}/data/partitions/{DELETE_PARTITION}"
print(f"Удаляем: {remote_path}")
print(f"  на клиентах: {[n['name'] for n in cfg['clients']]}")
print(f"  на сервере:  {cfg['server']['name']}")
print()

def delete_on_client(node):
    r = ssh_client(cfg, node["ip"],
                   f"rm -rf {remote_path} && echo deleted || echo failed",
                   timeout=30)
    ok = r.returncode == 0 and "deleted" in r.stdout
    return node["name"], "✓" if ok else f"✗ {r.stderr.strip()}"

with concurrent.futures.ThreadPoolExecutor() as ex:
    results = list(ex.map(delete_on_client, cfg["clients"]))

# Сервер (хранит test/ и manifest.json)
r = ssh_server(cfg, f"rm -rf {remote_path} && echo deleted || echo failed", timeout=30)
ok = r.returncode == 0 and "deleted" in r.stdout
results.append((cfg["server"]["name"], "✓" if ok else f"✗ {r.stderr.strip()}"))

for name, status in results:
    print(f"  {status} {name}")

---
## 4. Готовность к запуску FL

Проверяет что всё готово перед `flwr run . yandex-cloud`:

| Проверка | Что смотрим |
|---|---|
| **SuperLink** | tmux-сессия `superlink` жива на сервере |
| **SuperNode** | tmux-сессия `supernode` жива на каждом клиенте |
| **Партиция (клиент)** | `data/partitions/{name}/client_i/` существует |
| **Тест-сплит** | `data/partitions/{name}/test/` + `manifest.json` на сервере |

Имя партиции берётся из `pyproject.toml` автоматически.

In [ ]:
# 4. Preflight check — готовность к запуску FL
import concurrent.futures
import tomllib
from pathlib import Path

import pandas as pd
from IPython.display import display
from scripts.cluster_utils import ssh_client, ssh_server

# ── Читаем конфиг из pyproject.toml ──────────────────────────────────────
with open("pyproject.toml", "rb") as _f:
    _toml = tomllib.load(_f)
_fl_cfg = _toml["tool"]["flwr"]["app"]["config"]

PART_NAME   = _fl_cfg["partition-name"]
MODEL       = _fl_cfg["model"]
AGGREGATION = _fl_cfg["aggregation"]
NUM_ROUNDS  = _fl_cfg["num-server-rounds"]
MIN_TRAIN   = _fl_cfg["min-train-nodes"]
MIN_AVAIL   = _fl_cfg["min-available-nodes"]

REMOTE_DIR  = cfg["remote_dir"]
REMOTE_PART = f"{REMOTE_DIR}/data/partitions/{PART_NAME}"

print(f"Конфиг эксперимента из pyproject.toml:")
print(f"  partition-name      = {PART_NAME}")
print(f"  model               = {MODEL}")
print(f"  aggregation         = {AGGREGATION}")
print(f"  num-server-rounds   = {NUM_ROUNDS}")
print(f"  min-train-nodes     = {MIN_TRAIN}   min-available-nodes = {MIN_AVAIL}")
print()


# ── Проверки на сервере ───────────────────────────────────────────────────
def check_server() -> dict:
    row = {
        "Узел":       cfg["server"]["name"],
        "Роль":       "server",
        "SuperLink":  "—",
        "SuperNode":  "—",
        "Партиция":   "—",
    }

    # SuperLink: tmux-сессия + процесс (pgrep -f — по полной командной строке)
    r = ssh_server(cfg, "tmux list-sessions 2>/dev/null | grep -c superlink || echo 0", timeout=15)
    if r.returncode == 0 and r.stdout.strip() != "0":
        r2 = ssh_server(cfg, "pgrep -f flower-superlink > /dev/null && echo running || echo dead", timeout=10)
        row["SuperLink"] = "✓" if r2.returncode == 0 and "running" in r2.stdout else "⚠ tmux OK, process dead"
    else:
        row["SuperLink"] = "✗ not running"

    # Тест-сплит + manifest.json
    r = ssh_server(cfg,
        f"test -d {REMOTE_PART}/test && test -f {REMOTE_PART}/manifest.json && echo ok || echo missing",
        timeout=15)
    row["Партиция"] = "✓ test+manifest" if r.returncode == 0 and "ok" in r.stdout else "✗ missing"

    return row


# ── Проверки на клиентах ──────────────────────────────────────────────────
def check_client(node: dict) -> dict:
    i = node["partition_id"]
    row = {
        "Узел":       node["name"],
        "Роль":       f"client (pid={i})",
        "SuperLink":  "—",
        "SuperNode":  "—",
        "Партиция":   "—",
    }

    # SuperNode: tmux-сессия + процесс (pgrep -f — по полной командной строке)
    r = ssh_client(cfg, node["ip"],
        "tmux list-sessions 2>/dev/null | grep -c supernode || echo 0",
        timeout=15)
    if r.returncode == 0 and r.stdout.strip() != "0":
        r2 = ssh_client(cfg, node["ip"],
            "pgrep -f flower-supernode > /dev/null && echo running || echo dead",
            timeout=10)
        row["SuperNode"] = "✓" if r2.returncode == 0 and "running" in r2.stdout else "⚠ tmux OK, process dead"
    else:
        row["SuperNode"] = "✗ not running"

    # Партиция клиента
    r = ssh_client(cfg, node["ip"],
        f"test -d {REMOTE_PART}/client_{i} && echo ok || echo missing",
        timeout=15)
    row["Партиция"] = f"✓ client_{i}" if r.returncode == 0 and "ok" in r.stdout else f"✗ client_{i} missing"

    return row


# ── Запуск параллельно ────────────────────────────────────────────────────
print("Проверяем готовность кластера (параллельно)...")
with concurrent.futures.ThreadPoolExecutor() as ex:
    srv_future  = ex.submit(check_server)
    cli_futures = [ex.submit(check_client, node) for node in cfg["clients"]]
    rows = [srv_future.result()] + [f.result() for f in cli_futures]

df = pd.DataFrame(rows).set_index("Узел")
display(df)

# ── Итоговый вердикт ──────────────────────────────────────────────────────
superlink_ok  = rows[0]["SuperLink"].startswith("✓")
supernodes_ok = all(r["SuperNode"].startswith("✓") for r in rows[1:])
partitions_ok = all(r["Партиция"].startswith("✓") for r in rows)

print()
checks = [
    ("SuperLink",                superlink_ok),
    (f"SuperNodes ({len(cfg['clients'])})",  supernodes_ok),
    ("Партиции на всех узлах",   partitions_ok),
]
all_ok = True
for label, ok in checks:
    icon = "✓" if ok else "✗"
    print(f"  {icon} {label}")
    if not ok:
        all_ok = False

print()
if all_ok:
    print("✓ Всё готово. Запускай:")
    print(f"    source .venv/bin/activate && flwr run . yandex-cloud")
else:
    print("✗ Не готово. Что сделать:")
    if not superlink_ok:
        print("  • SuperLink:  ssh -i ~/.ssh/admin-fl gleb@84.201.165.255 'bash ~/ddl/deploy/start_superlink.sh'")
    if not supernodes_ok:
        print("  • SuperNodes: bash deploy/start_supernodes.sh")
    if not partitions_ok:
        print(f"  • Партиции:   выполни ячейку 3.3 (rsync) для '{PART_NAME}'")

### 4.1 Запуск SuperLink (сервер)

In [ ]:
# 4.1 — Запуск SuperLink на сервере
from scripts.cluster_utils import ssh_server

r = ssh_server(cfg, "bash ~/ddl/deploy/start_superlink.sh", timeout=15)
print(r.stdout.strip() or r.stderr.strip())

### 4.2 Запуск SuperNodes (все клиенты)

In [ ]:
# 4.2 — Запуск SuperNodes на всех клиентах
import concurrent.futures
from scripts.cluster_utils import ssh_client

# Клиенты подключаются к SuperLink по внутреннему IP (VPC)
SERVER_INT     = cfg["server_int"]
NUM_PARTITIONS = len(cfg["clients"])
REMOTE_DIR     = cfg["remote_dir"]
SESSION        = "supernode"

def start_supernode(node: dict) -> tuple[str, str]:
    i = node["partition_id"]
    # Все TOML-пары передаются в ОДНОМ --node-config через пробел
    node_cfg = f"partition-id={i} num-partitions={NUM_PARTITIONS}"
    cmd = (
        f"tmux kill-session -t {SESSION} 2>/dev/null || true && "
        f"tmux new-session -d -s {SESSION} "
        f"\"bash -c 'cd {REMOTE_DIR} && source .venv/bin/activate && "
        f"exec flower-supernode --superlink {SERVER_INT}:9092 --insecure "
        f"--node-config \\\"{node_cfg}\\\"'\" && "
        f"echo started"
    )
    r = ssh_client(cfg, node["ip"], cmd, timeout=20)
    ok = r.returncode == 0 and "started" in r.stdout
    return node["name"], "✓ started" if ok else f"✗ {r.stderr.strip() or r.stdout.strip()}"

print(f"Запускаем {NUM_PARTITIONS} SuperNodes → SuperLink {SERVER_INT}:9092 (параллельно)...")
with concurrent.futures.ThreadPoolExecutor() as ex:
    results = list(ex.map(start_supernode, cfg["clients"]))

for name, status in results:
    print(f"  {status}  {name}")

---
## 5. Запуск FL-эксперимента

Запускает `flwr run . yandex-cloud` и стримит логи прямо в ячейку.  
Прервать обучение — **Kernel → Interrupt** (или кнопка ■ в тулбаре).

In [ ]:
# 5. Запуск FL-эксперимента
import subprocess
import os
import re
from pathlib import Path

flwr_bin = Path(".venv/bin/flwr")
env = os.environ.copy()
env["PYTHONUNBUFFERED"] = "1"

# ── Запуск ───────────────────────────────────────────────────────────────
print("$ flwr run . yandex-cloud")
r = subprocess.run(
    [str(flwr_bin), "run", ".", "yandex-cloud"],
    capture_output=True, text=True, env=env,
)
output = r.stdout + r.stderr
print(output.strip())

# ── Извлекаем run-id: "Successfully started run 1234567890"
match = re.search(r"run\s+(\d+)", output, re.IGNORECASE)
if not match:
    print("✗ Не удалось получить run-id. Используй ячейку 5.2 чтобы найти его вручную.")
else:
    run_id = match.group(1)
    print(f"\nRun ID: {run_id}")
    print("─" * 70)

    log_proc = subprocess.Popen(
        [str(flwr_bin), "log", run_id, ".", "yandex-cloud"],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1, env=env,
    )
    try:
        for line in log_proc.stdout:
            print(line, end="", flush=True)
    finally:
        log_proc.wait()
    print("─" * 70)
    print(f"Завершено. Exit code: {log_proc.returncode}")

In [ ]:
# 5.1 — Остановить FL (SuperLink + SuperNodes на всех узлах)
# Запускай если нужно принудительно завершить обучение.
# Если ячейка 5 ещё выполняется — сначала прерви её: Kernel → Interrupt (кнопка ■)
import concurrent.futures
from scripts.cluster_utils import ssh_server, ssh_client

STOP_CMD = """
    tmux kill-session -t superlink 2>/dev/null || true
    tmux kill-session -t supernode 2>/dev/null || true
    pkill -f flower-superlink 2>/dev/null || true
    pkill -f flower-supernode 2>/dev/null || true
    echo done
"""

def stop_node(node):
    if node["role"] == "server":
        r = ssh_server(cfg, STOP_CMD, timeout=15)
    else:
        r = ssh_client(cfg, node["ip"], STOP_CMD, timeout=15)
    ok = r.returncode == 0 and "done" in r.stdout
    return node["name"], "✓ stopped" if ok else f"✗ {r.stderr.strip()}"

print("Останавливаем все Flower процессы (параллельно)...")
with concurrent.futures.ThreadPoolExecutor() as ex:
    results = list(ex.map(stop_node, cfg["all_nodes"]))

for name, status in results:
    print(f"  {status}  {name}")

In [ ]:
# 5.2 — Список запусков и подключение к логам вручную
import subprocess
import os
from pathlib import Path

flwr_bin = Path(".venv/bin/flwr")
env = os.environ.copy()
env["PYTHONUNBUFFERED"] = "1"

# ── Список всех запусков ─────────────────────────────────────────────────
print("$ flwr ls . yandex-cloud --runs")
r = subprocess.run(
    [str(flwr_bin), "ls", ".", "yandex-cloud", "--runs"],
    capture_output=True, text=True, env=env,
)
print(r.stdout or r.stderr)

# ==============================================================================
# Вставь run-id из таблицы выше
RUN_ID = None   # ← например: RUN_ID = 42
# ==============================================================================

if RUN_ID is not None:
    print(f"─" * 70)
    log_proc = subprocess.Popen(
        [str(flwr_bin), "log", str(RUN_ID), ".", "yandex-cloud"],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1, env=env,
    )
    try:
        for line in log_proc.stdout:
            print(line, end="", flush=True)
    finally:
        log_proc.wait()

---
## 6. Артефакты экспериментов

**6.1** — список экспериментов на сервере  
**6.2** — скачать эксперимент на локальную машину в `runs/`

In [ ]:
# 6.1 — Список экспериментов на сервере
import json
from scripts.cluster_utils import ssh_server

import pandas as pd
from IPython.display import display

REMOTE_RUNS = f"{cfg['remote_dir']}/runs"

r = ssh_server(cfg, f"""
python3 - << 'PYEOF'
import os, json, glob

runs_dir = os.path.expanduser("{REMOTE_RUNS}")
if not os.path.isdir(runs_dir):
    print(json.dumps([]))
else:
    rows = []
    for exp in sorted(os.listdir(runs_dir)):
        exp_path = os.path.join(runs_dir, exp)
        if not os.path.isdir(exp_path):
            continue
        summaries = glob.glob(os.path.join(exp_path, "*__summary.json"))
        if summaries:
            with open(sorted(summaries)[-1]) as f:
                s = json.load(f)
            rows.append({{
                "experiment": exp,
                "rounds": s.get("num_server_rounds", "—"),
                "best_acc": f"{{s.get('best_acc', 0)*100:.2f}}%",
                "best_round": s.get("best_round", "—"),
                "comm_mb": f"{{s.get('total_comm_mb', 0):.1f}}",
            }})
        else:
            # папка есть, но summary ещё нет (обучение не завершено)
            rows.append({{"experiment": exp, "rounds": "—", "best_acc": "—", "best_round": "—", "comm_mb": "—"}})
    print(json.dumps(rows))
PYEOF
""", timeout=20)

if r.returncode != 0:
    print(f"Ошибка: {r.stderr.strip()}")
else:
    rows = json.loads(r.stdout.strip())
    if not rows:
        print(f"Папка {REMOTE_RUNS} пуста или не существует.")
    else:
        df = pd.DataFrame(rows).set_index("experiment")
        df.columns = ["Раундов", "Best Acc", "Best Round", "Comm MB"]
        display(df)
        print(f"\nВсего экспериментов: {len(rows)}")

In [ ]:
# 6.2 — Скачать эксперимент с сервера
# Имя эксперимента можно скопировать из таблицы выше
from pathlib import Path
from scripts.cluster_utils import rsync_from_server

# ==============================================================================
EXPERIMENT = "cifar10__dirichlet__a0.8__m50__n6__s42__cifarcnn__fedavg__7ed8"  # ← МЕНЯЙ ЗДЕСЬ
# ==============================================================================

src = f"{cfg['remote_dir']}/runs/{EXPERIMENT}/"
dst = f"runs/{EXPERIMENT}/"

Path(dst).mkdir(parents=True, exist_ok=True)
print(f"Скачиваем: {cfg['server']['name']}:{src}")
print(f"      в  : {Path(dst).resolve()}")
print()

r = rsync_from_server(cfg, src, dst, timeout=120)

if r.returncode == 0:
    files = sorted(Path(dst).iterdir())
    print(f"✓ Готово. Файлов: {len(files)}")
    for f in files:
        size_kb = f.stat().st_size / 1024
        print(f"  {f.name:<50} {size_kb:>8.1f} KB")
else:
    print(f"✗ rsync завершился с ошибкой (code={r.returncode})")

---
## 7. Анализ результатов эксперимента

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from IPython.display import display, HTML

# ==============================================================================
EXP_NAME = "cifar10__dirichlet__a0.9__m50__n6__s42__cifarcnn__fedavg__6da2"  # ← МЕНЯЙ ЗДЕСЬ
# ==============================================================================

# ── Вспомогательная функция: таблица с рамкой ─────────────────────────────
def _box_table(headers, rows, title=None):
    """Печатает таблицу с Unicode-рамкой."""
    widths = [len(str(h)) for h in headers]
    for row in rows:
        for i, v in enumerate(row):
            widths[i] = max(widths[i], len(str(v)))

    def _border(l, m, r):
        return l + m.join("─" * (w + 2) for w in widths) + r

    def _row(cells):
        return "│" + "│".join(f" {str(v):<{widths[i]}} " for i, v in enumerate(cells)) + "│"

    if title:
        print(f"\n{title}")
    print(_border("┌", "┬", "┐"))
    print(_row(headers))
    print(_border("├", "┼", "┤"))
    for i, row in enumerate(rows):
        print(_row(row))
        if i < len(rows) - 1:
            print(_border("├", "┼", "┤"))
    print(_border("└", "┴", "┘"))


exp_dir   = Path("runs") / EXP_NAME
prof_path = exp_dir / "cluster_profile.json"

# Имя партиции и названия классов берём из summary.json
summaries  = sorted(exp_dir.glob("*__summary.json"))
_summary   = json.loads(summaries[-1].read_text()) if summaries else {}
_part_name = _summary.get("config", {}).get("partition_name", "")
_cls_names = _summary.get("config", {}).get("class_names", [])

# ── Профиль кластера (опционально) ────────────────────────────────────────
if not prof_path.exists():
    print(f"⚠  cluster_profile.json не найден — эксперимент запущен без enable-profiling=true")
    print(f"   Таблицы 1–4 (профиль кластера) будут пропущены.")
    print(f"   Метрики обучения (ячейка ниже) доступны без профиля.\n")
    profile   = None
    clients   = {}
    cids      = []
    n_clients = 0
else:
    profile   = json.loads(prof_path.read_text())
    clients   = profile["clients"]
    cids      = sorted(clients.keys(), key=int)
    n_clients = len(cids)

    print(f"Профиль кластера — {profile['partition_name']}")
    print(f"{n_clients} клиентов | профилировочный раунд (1000 сэмплов, 2 эпохи)\n")

    # ══════════════════════════════════════════════════════════════════════════
    # Таблица 1: Аппаратные характеристики
    # ══════════════════════════════════════════════════════════════════════════
    _box_table(
        ["Client", "CPU (лог.)", "CPU (физ.)", "МГц", "RAM GB", "RAM своб."],
        [
            [f"client_{cid}",
             int(clients[cid].get("hw_cpu_logical",  0)),
             int(clients[cid].get("hw_cpu_physical", 0)),
             int(clients[cid].get("hw_cpu_freq_mhz", 0)),
             round(clients[cid].get("hw_ram_total_gb", 0), 1),
             round(clients[cid].get("hw_ram_avail_gb", 0), 1)]
            for cid in cids
        ],
        title="1. Аппаратные характеристики",
    )

    # ══════════════════════════════════════════════════════════════════════════
    # Таблица 2: Распределение данных
    # ══════════════════════════════════════════════════════════════════════════
    _box_table(
        ["Client", "Сэмплов", "Классов", "Энтропия", "Макс. кл.", "Мин. кл.", "Вырожд.", "Отсутств."],
        [
            [f"client_{cid}",
             int(clients[cid].get("data_num_samples",      0)),
             int(clients[cid].get("data_n_classes",         0)),
             f"{clients[cid].get('entropy_norm', 0.0):.4f}",
             int(clients[cid].get("data_max_class_count",   0)),
             int(clients[cid].get("data_min_class_count",   0)),
             ", ".join(str(c) for c in clients[cid].get("degenerate_classes", [])) or "—",
             ", ".join(str(c) for c in clients[cid].get("missing_classes",    [])) or "—"]
            for cid in cids
        ],
        title="2. Распределение данных",
    )

    # Системная метрика гетерогенности
    mpjs = profile.get("mean_pairwise_js")
    if mpjs is not None:
        label = "близко к IID" if mpjs < 0.05 else "умеренная" if mpjs < 0.20 else "высокая"
        print(f"\n  Межклиентская гетерогенность (mean pairwise JS): {mpjs:.4f}  ({label})")

    # ══════════════════════════════════════════════════════════════════════════
    # Таблица 2a: Серверный датасет (если был использован)
    # ══════════════════════════════════════════════════════════════════════════
    srv = profile.get("server_schedule")
    if srv:
        js_before = srv.get("mean_pairwise_js_before", 0.0)
        js_after  = srv.get("mean_pairwise_js_after",  0.0)
        js_red    = srv.get("js_reduction_pct", 0.0)
        mode      = srv.get("mode", "?")
        per_cls   = srv.get("per_class", 0)
        srv_total = srv.get("server_size", 0)
        srv_cls   = srv.get("clients", {})

        srv_rows = []
        for cid in cids:
            client_srv = srv_cls.get(str(cid), {})
            total_s = client_srv.get("total_server_samples", 0)
            counts  = client_srv.get("class_counts", [])
            top3    = sorted(range(len(counts)), key=lambda k: -counts[k])[:3]
            top3_str = "  ".join(
                f"{(_cls_names[int(k)] if _cls_names and int(k) < len(_cls_names) else k)}:{counts[k]}"
                for k in top3 if counts[k] > 0
            ) or "—"
            srv_rows.append([f"client_{cid}", total_s, top3_str])

        _box_table(
            ["Client", "Сэмплов с сервера", "Топ-3 классов (дефицит → сервер)"],
            srv_rows,
            title=f"2a. Серверный датасет  (mode={mode}, server_size={srv_total}, {per_cls}/class)",
        )
        sign = "↓" if js_red > 0 else ("↑" if js_red < 0 else "=")
        print(
            f"  JS гетерогенность: {js_before:.4f} → {js_after:.4f}  "
            f"({sign} {abs(js_red):.1f}%)"
        )

    # ══════════════════════════════════════════════════════════════════════════
    # Таблица 3: Бенчмарк скорости
    # ══════════════════════════════════════════════════════════════════════════
    all_sps = [clients[c].get("bench_samples_per_sec", 1) for c in cids]
    max_sps = max(all_sps) if all_sps else 1
    _box_table(
        ["Client", "Сэмпл/сек", "s / 1k", "Эпоха (s)", "Bench loss", "Отн."],
        [
            [f"client_{cid}",
             round(clients[cid].get("bench_samples_per_sec", 0), 1),
             round(clients[cid].get("bench_time_per_1k_sec", 0), 3),
             round(clients[cid].get("bench_mean_epoch_sec",  0), 2),
             round(clients[cid].get("bench_final_loss",      0), 4),
             f"{clients[cid].get('bench_samples_per_sec', 0) / max_sps:.2f}×"]
            for cid in cids
        ],
        title="3. Бенчмарк скорости",
    )
    fastest = max(all_sps)
    slowest = min(all_sps)
    if slowest > 0:
        print(f"\n  Straggler gap: {fastest:.0f} s/s → {slowest:.0f} s/s = {fastest/slowest:.1f}× разница")

    # ══════════════════════════════════════════════════════════════════════════
    # Heatmap: распределение классов по клиентам
    # ══════════════════════════════════════════════════════════════════════════
    class_keys = set()
    for d in clients.values():
        class_keys |= set(d.get("class_distribution", {}).keys())
    all_class_ids    = sorted(class_keys, key=int)
    n_classes_actual = len(all_class_ids)

    if n_classes_actual > 0:
        matrix = np.zeros((n_classes_actual, n_clients), dtype=int)
        for j, cid in enumerate(cids):
            dist = clients[cid].get("class_distribution", {})
            for i, cls_id in enumerate(all_class_ids):
                matrix[i, j] = dist.get(str(cls_id), dist.get(cls_id, 0))

        row_labels = (
            [f"{cls_id}: {_cls_names[int(cls_id)]}" for cls_id in all_class_ids]
            if _cls_names and len(_cls_names) >= n_classes_actual
            else [str(c) for c in all_class_ids]
        )

        print("\n4. Распределение классов по клиентам")
        fig, ax = plt.subplots(figsize=(max(6, n_clients * 1.4), max(4, n_classes_actual * 0.45)))
        im = ax.imshow(matrix, aspect="auto", cmap="Blues", vmin=0)
        plt.colorbar(im, ax=ax, label="Сэмплов")
        ax.set_xticks(range(n_clients))
        ax.set_xticklabels([f"client_{c}" for c in cids], rotation=30, ha="right")
        ax.set_yticks(range(n_classes_actual))
        ax.set_yticklabels(row_labels, fontsize=8)
        ax.set_title(f"Распределение классов — {_part_name}", fontsize=10)
        thresh = matrix.max() * 0.6
        fs = 7 if n_classes_actual <= 20 else 5
        for r in range(n_classes_actual):
            for c in range(n_clients):
                v = matrix[r, c]
                ax.text(c, r, f"{v:,}", ha="center", va="center", fontsize=fs,
                        color="white" if v > thresh else "black")
        plt.tight_layout()
        plt.show()

In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from pathlib import Path

# EXP_NAME и _box_table задаются в ячейке выше (Профиль кластера).
# Если запускаешь эту ячейку отдельно — задай здесь:
# EXP_NAME = "cifar10__iid__n6__s42__cifarcnn__fedavg__4060"

exp_dir = Path("runs") / EXP_NAME
candidates = sorted(exp_dir.glob("*r__summary.json"))
if not candidates:
    print(f"✗ Нет данных в {exp_dir}"); raise SystemExit(1)

summary_path = candidates[-1]
prefix = summary_path.name.replace("__summary.json", "")

summary = json.loads(summary_path.read_text())
df_r    = pd.read_csv(exp_dir / f"{prefix}__rounds.csv")
df_c    = pd.read_csv(exp_dir / f"{prefix}__clients.csv")
cfg     = summary["config"]

TIME_COL      = "round_total_time_sec" if "round_total_time_sec" in df_r.columns else "round_wall_time_sec"
HAS_F1        = "test_f1" in df_r.columns
HAS_SERVER_JS = "effective_js" in df_r.columns and df_r["effective_js"].max() > 0.0

print(f"{'═'*68}")
print(f"  {EXP_NAME}")
print(f"  {prefix} · {cfg['dataset'].upper()} · {cfg['scheme']} · {cfg['num_clients']} clients")
adaptive_info = ""
if cfg.get("enable_profiling") is not None:
    if cfg["enable_profiling"]:
        adaptive_info = f"  adaptive: {cfg.get('adaptive_mode', '—')}"
    else:
        adaptive_info = "  adaptive: disabled"
if adaptive_info:
    print(adaptive_info)
server_mode = cfg.get("server_mode", "disabled")
if server_mode and server_mode != "disabled":
    print(f"  server-mode: {server_mode}")
print(f"{'═'*68}\n")

df_active = df_r[df_r["round"] > 0]
total_time = summary.get("total_wall_time_sec", df_active[TIME_COL].sum())
best_acc   = summary.get("best_test_acc", df_r["test_acc"].max())
best_rnd   = summary.get("best_round",    int(df_r.loc[df_r["test_acc"].idxmax(), "round"]))
best_f1    = summary.get("best_test_f1",  df_r["test_f1"].max() if HAS_F1 else None)
best_f1_r  = summary.get("best_f1_round", int(df_r.loc[df_r["test_f1"].idxmax(), "round"]) if HAS_F1 else None)
final_acc  = df_r.iloc[-1]["test_acc"]
final_f1   = df_r.iloc[-1]["test_f1"] if HAS_F1 else None
final_loss = df_r.iloc[-1]["test_loss"]
mean_round = df_active[TIME_COL].mean()
mean_strag = df_active.eval("max_client_time_sec - min_client_time_sec").mean()
mean_drift = df_active["mean_drift"].mean()

# ── 1. Метрики качества модели ────────────────────────────────────────────
quality_rows = [
    ["Best accuracy",   f"{best_acc*100:.2f}%",                            f"round {best_rnd}"],
    ["Final accuracy",  f"{final_acc*100:.2f}%",                           f"round {int(df_r.iloc[-1]['round'])}"],
]
if HAS_F1:
    quality_rows += [
        ["Best macro F1",   f"{best_f1*100:.2f}%",    f"round {best_f1_r}"],
        ["Final macro F1",  f"{final_f1*100:.2f}%",   f"round {int(df_r.iloc[-1]['round'])}"],
    ]
quality_rows += [
    ["Final test loss", f"{final_loss:.4f}",                               ""],
    ["Rounds to 80%",   str(summary.get("rounds_to_80pct") or "—"),        ""],
    ["Rounds to 90%",   str(summary.get("rounds_to_90pct") or "—"),        ""],
]
if HAS_SERVER_JS:
    eff_js = df_r["effective_js"].iloc[0]
    quality_rows += [
        ["Effective JS (after server data)", f"{eff_js:.4f}", f"server-mode={server_mode}"],
    ]
_box_table(
    ["Метрика", "Значение", "Примечание"],
    quality_rows,
    title="1. Метрики качества модели",
)

# ── 2. Метрики эффективности FL ───────────────────────────────────────────
_box_table(
    ["Параметр", "Значение"],
    [
        ["Раундов",              str(cfg["num_rounds"])],
        ["Клиентов",             str(cfg["num_clients"])],
        ["Локальных эпох",       str(cfg["local_epochs"])],
        ["Стратегия",            cfg["aggregation"]],
        ["Всего трафика",        f"{summary.get('total_comm_mb', 0):.2f} MB"],
        ["Трафик / раунд",       f"{summary.get('total_comm_mb', 0) / max(cfg['num_rounds'], 1):.2f} MB"],
        ["Общее время",          f"{total_time:.0f}s ({total_time/60:.1f} мин)"],
        ["Среднее время раунда", f"{mean_round:.1f}s"],
        ["Straggler gap (avg)",  f"{mean_strag:.1f}s"],
        ["Mean client drift",    f"{mean_drift:.4f}"],
    ],
    title="2. Метрики эффективности FL",
)

# ── 2.1 Разбивка времени раунда ───────────────────────────────────────────
if "train_time_sec" in df_r.columns:
    t_train = df_active["train_time_sec"].mean()
    t_agg   = df_active["agg_time_sec"].mean()
    t_eval  = df_active["eval_time_sec"].mean()
    t_total = t_train + t_agg + t_eval
    _box_table(
        ["Компонент", "Среднее", "Доля"],
        [
            ["Ожидание клиентов (train)", f"{t_train:.1f}s", f"{t_train/t_total*100:.1f}%"],
            ["Агрегация весов (agg)",     f"{t_agg:.2f}s",   f"{t_agg/t_total*100:.1f}%"],
            ["Evaluate на тесте (eval)",  f"{t_eval:.2f}s",  f"{t_eval/t_total*100:.1f}%"],
            ["Итого (round_total)",       f"{t_total:.1f}s", "100%"],
        ],
        title="2.1 Разбивка времени раунда",
    )

# ── 3. Таблица раундов ────────────────────────────────────────────────────
rounds_headers = ["Round", "Test Acc"]
if HAS_F1:
    rounds_headers.append("F1")
rounds_headers += ["Δ Acc", "Test Loss", "Train Loss", "Drift", "Slowest", "Comm MB"]
if HAS_SERVER_JS:
    rounds_headers.append("Eff. JS")

rounds_rows = []
for _, row in df_r.iterrows():
    r = int(row["round"])
    cols = [r, f"{row['test_acc']*100:.2f}%"]
    if HAS_F1:
        cols.append(f"{row['test_f1']*100:.2f}%")
    cols += [
        f"{row['delta_acc']*100:+.2f}%",
        f"{row['test_loss']:.4f}",
        f"{row['mean_train_loss']:.4f}",
        f"{row['mean_drift']:.3f}",
        f"{row['max_client_time_sec']:.1f}s",
        f"{row['cum_comm_mb']:.2f}",
    ]
    if HAS_SERVER_JS:
        cols.append(f"{row['effective_js']:.4f}")
    rounds_rows.append(cols)

_box_table(rounds_headers, rounds_rows, title="3. По раундам")

# ── 4. Straggler-эффект ───────────────────────────────────────────────────
agg_dict = {
    "mean_time": ("round_time_sec", "mean"),
    "max_time":  ("round_time_sec", "max"),
    "min_time":  ("round_time_sec", "min"),
}
strag = df_c.groupby("client_id").agg(**agg_dict).sort_values("mean_time")
fastest_t = strag["mean_time"].min()

strag_rows = []
for cid, row in strag.iterrows():
    strag_rows.append([
        f"client_{int(cid)}",
        f"{row['mean_time']:.1f}s",
        f"{row['max_time']:.1f}s",
        f"{row['min_time']:.1f}s",
        f"{row['mean_time'] / fastest_t:.2f}×",
    ])
_box_table(
    ["Client", "Среднее", "Макс", "Мин", "Замедление"],
    strag_rows,
    title="4. Straggler-эффект (по клиентам)",
)
straggler_gap = df_active.eval("max_client_time_sec - min_client_time_sec").mean()
print(f"  Средний straggler gap: {straggler_gap:.1f}s")

# ── 5. Графики ────────────────────────────────────────────────────────────
plots = [
    (f"{prefix}__accuracy.png",          "Accuracy & Loss"),
    (f"{prefix}__f1.png",                "F1 & Loss"),
    (f"{prefix}__train_loss_boxplot.png", "Train Loss Boxplot"),
    (f"{prefix}__class_accuracy.png",     "Per-Class Accuracy Heatmap"),
]
existing = [(fn, title) for fn, title in plots if (exp_dir / fn).exists()]

if existing:
    fig, axes = plt.subplots(1, len(existing), figsize=(7 * len(existing), 5))
    if len(existing) == 1:
        axes = [axes]
    for ax, (fn, title) in zip(axes, existing):
        img = mpimg.imread(exp_dir / fn)
        ax.imshow(img)
        ax.axis("off")
        ax.set_title(title, fontsize=11)
    plt.tight_layout()
    plt.show()